In [178]:
def pp(msg=None, value=None):
    if msg is not None:
        print(msg)
    if value is not None:
        print(value)
    print('=' * 100)

In [179]:
import pandas as pd

flights_raw = pd.read_csv('data/flights.csv')
flights = flights_raw.copy()
pp("flights.csv", flights.shape)

pp("Columns", flights.columns)
pp("Dtypes", flights.dtypes)

flights.csv
(20000, 5)
Columns
Index(['date', 'delay', 'distance', 'origin', 'destination'], dtype='str')
Dtypes
date             str
delay          int64
distance       int64
origin           str
destination      str
dtype: object


Every row is a single flight from origin to destination airport:

In [180]:
pp("Sample flight", flights.head(1))

Sample flight
                  date  delay  distance origin destination
0  2001-01-01 00:47:00     66      1750    DTW         LAS


# Data cleaning
Data problems:
1. `date` as string
2. `distance` unit not specified 
3. `delay` has negative values

In [181]:
# 1. `date` as string
# Convert `date` str to datetime using Pandas (20000 rows):
flights['date'] = pd.to_datetime(flights['date'])
flights.dtypes


date           datetime64[us]
delay                   int64
distance                int64
origin                    str
destination               str
dtype: object

In [182]:
# 2. `distance` unit not specified 
# Considering USA flight data, we can assume miles as distance unit.
# Let's make a quick check of distances for first 3 flights (https://www.greatcirclemap.com/):
# DTW -> LAS = 1749 miles
# HNL -> SFO = 2398 miles
# LAS -> OAK = 407 miles
# for distance in flights.head(3)['distance']:
#     print(distance)
# # 1750
# # 2399
# # 407
# As assumption is correct, let's rename the column:
flights.rename(columns={'distance': 'distance_miles'}, inplace=True)
flights.columns

Index(['date', 'delay', 'distance_miles', 'origin', 'destination'], dtype='str')

In [183]:
# 3. `delay` has negative values
# Negative/zero delay means early/no delay arrival.
# A separate boolean value can be useful for further evaluations (20000 rows):
flights['is_delayed'] = flights['delay'] > 0
pp("Total of flights", len(flights))
flights['is_delayed'].value_counts(normalize=True)

Total of flights
20000


is_delayed
False    0.52535
True     0.47465
Name: proportion, dtype: float64

In [184]:
# As the description says that `delay` is in minutes, let's adjust the corresponding column name:
flights.rename(columns={'delay': 'delay_minutes'}, inplace=True)
# And save cleaned data:
flights.to_csv('data/flights_clean.csv', index=False)
flights.dtypes

date              datetime64[us]
delay_minutes              int64
distance_miles             int64
origin                       str
destination                  str
is_delayed                  bool
dtype: object

# Missing values analysis

There is NO missing values in columns (i.e. we keep everything).
Though, `delay` and `distanse` column names have been renamed in previous steps to show units:
`delay` -> `delay_minutes`
and
`distance` -> `distance_miles`.
This is important for furter analysis, e.g. for `longer distance -> higher delay` correlation.
Without understanding the units of measure, it can be difficult to evaluate

In [185]:

print(flights.isna().sum())

date              0
delay_minutes     0
distance_miles    0
origin            0
destination       0
is_delayed        0
dtype: int64


# Merging tables (Flights and Airports data)
For further analysis flights.csv and airports.csv can be merged on `iata` of each airport.
As a result additional airport data will be available: name, city, state, country, latitude and longitude:

In [186]:
airports_raw = pd.read_csv("data/airports.csv")
airports = airports_raw.copy()
pp("flights.csv", flights.shape)
pp(airports.dtypes)

pp("Flights before merge", len(flights))
pp("Airports", len(airports))

origin_airports = airports.rename(columns={
    "iata": "origin",
    "name": "origin_airport_name",
    "city": "origin_city",
    "state": "origin_state",
    "country": "origin_country",
    "latitude": "origin_latitude",
    "longitude": "origin_longitude"
})

destination_airports = airports.rename(columns={
    "iata": "destination",
    "name": "destination_airport_name",
    "city": "destination_city",
    "state": "destination_state",
    "country": "destination_country",
    "latitude": "destination_latitude",
    "longitude": "destination_longitude"
})

merged = flights.merge(
    origin_airports,
    on="origin",
    how="left",
    validate="many_to_one"
)
pp("After origin merge", len(merged))

merged = merged.merge(
    destination_airports,
    on="destination",
    how="left",
    validate="many_to_one"
)
pp("After destination merge", len(merged))

pp("Missing origin airports", merged["origin_airport_name"].isna().sum())
pp("Missing destination airports", merged["destination_airport_name"].isna().sum())
pp("Duplicated rows", merged.duplicated().sum())

merged.to_csv('data/merged.csv', index=False)
pp("merged.csv", merged.shape)
pp(merged.dtypes)

flights.csv
(20000, 6)
iata             str
name             str
city             str
state            str
country          str
latitude     float64
longitude    float64
dtype: object
Flights before merge
20000
Airports
3376
After origin merge
20000
After destination merge
20000
Missing origin airports
0
Missing destination airports
0
Duplicated rows
0
merged.csv
(20000, 18)
date                        datetime64[us]
delay_minutes                        int64
distance_miles                       int64
origin                                 str
destination                            str
is_delayed                            bool
origin_airport_name                    str
origin_city                            str
origin_state                           str
origin_country                         str
origin_latitude                    float64
origin_longitude                   float64
destination_airport_name               str
destination_city                       str
destination_state   

# Adding new columns
We already added `is_delayed` column, second useful one can be `weekday` of a delay (e.g. it can be used for evaluation of most common days of flights delay): 

In [187]:
merged['hour'] = pd.to_datetime(merged['date']).dt.hour
pp(merged['hour'].value_counts())

merged['weekday'] = pd.to_datetime(merged['date']).dt.day_name()
pp(merged['weekday'].value_counts())
# pp(merged['weekday'].value_counts(normalize=True).round(2))

merged.to_csv('data/merged.csv', index=False)
pp("merged.csv", merged.shape)
pp(merged.dtypes)

hour
17    1374
8     1323
6     1313
13    1307
7     1280
9     1266
19    1244
12    1229
11    1225
18    1206
16    1165
15    1156
14    1117
10    1094
20     988
21     711
22     456
5      240
23     171
0       70
1       53
2        9
3        2
4        1
Name: count, dtype: int64
weekday
Friday       3003
Monday       2952
Thursday     2944
Tuesday      2903
Wednesday    2895
Saturday     2655
Sunday       2648
Name: count, dtype: int64
merged.csv
(20000, 20)
date                        datetime64[us]
delay_minutes                        int64
distance_miles                       int64
origin                                 str
destination                            str
is_delayed                            bool
origin_airport_name                    str
origin_city                            str
origin_state                           str
origin_country                         str
origin_latitude                    float64
origin_longitude                   float64
destin

# Group and compare
Group summaries for delay big picture:

In [188]:
delayed = merged[merged['is_delayed']]

In [189]:
worst_delays = (
    delayed.groupby(['origin', 'destination'])
    .agg(
        delayed_flights=('delay_minutes', 'count'),
        average_delay=('delay_minutes', 'mean'),
        median_delay=('delay_minutes', 'median')
    )
    .sort_values(
        by=['delayed_flights', 'average_delay'], 
        ascending=[False, False]
    )
)
pp("Worst routes (top 10)", worst_delays.head(10))

Worst routes (top 10)
                    delayed_flights  average_delay  median_delay
origin destination                                              
PHX    LAS                       33      30.212121          11.0
       LAX                       32      29.812500          21.5
LAX    PHX                       30      25.633333          13.5
LAS    LAX                       29      25.413793          17.0
       PHX                       29      16.275862           6.0
LAX    LAS                       28      36.178571          19.5
SFO    LAX                       24      34.958333          20.0
DFW    ORD                       23      41.478261          17.0
LAX    SJC                       23      35.130435          23.0
PHX    SAN                       23      27.565217          10.0


In [190]:
airport_delays = (
    delayed.groupby('origin')
    .agg(
        delayed_flights=('delay_minutes', 'count'),
        # average_delay=('delay_minutes', 'mean'),
        # median_delay=('delay_minutes', 'median'),
        total_delay=('delay_minutes', 'sum'),
    )
    .query('delayed_flights > 99')
    .sort_values('delayed_flights', ascending=False)
)
pp("Worst airports (top 10) with more than 99 entries", airport_delays.head(10))

Worst airports (top 10) with more than 99 entries
        delayed_flights  total_delay
origin                              
DFW                 542        15802
ORD                 493        14910
ATL                 422         9830
LAX                 382        11019
PHX                 372         9700
STL                 297         7292
LAS                 241         6617
DEN                 237         7118
EWR                 211         6143
CLT                 206         5008


In [191]:
hourly_delays = (
    delayed.groupby('hour')
    .agg(
        delayed_flights=('delay_minutes', 'count'),
        average_delay=('delay_minutes', 'mean')
    )
    .sort_values('delayed_flights', ascending=False)
)
pp("Delayed flights by hour (top 10)", hourly_delays.head(7))

weekday_delays = (
    delayed.groupby('weekday')
    .agg(
        delayed_flights=('delay_minutes', 'count'),
        average_delay=('delay_minutes', 'mean')
    )
    .sort_values('delayed_flights', ascending=False)
)
pp("Delayed flights by weekday (top 10)", weekday_delays)

Delayed flights by hour (top 10)
      delayed_flights  average_delay
hour                                
17                731      26.993160
19                664      34.625000
13                627      23.470494
12                613      27.662316
18                612      30.736928
15                594      25.828283
16                578      28.647059
Delayed flights by weekday (top 10)
           delayed_flights  average_delay
weekday                                  
Friday                1649      28.223166
Thursday              1509      26.717694
Wednesday             1364      25.277126
Tuesday               1336      27.047156
Monday                1319      27.064443
Sunday                1219      26.580804
Saturday              1097      24.580675


# Describe with numbers (delay_minutes)
**median** (50%) shows that half of flights are on time or have early arrivals;

**spread** (std) is about 31% which means that there is wide variaty of values;

**outliers** (max, min) are flights with late arrival by 522 minutes and early arrival by -59 minutes, though they are exceptions - they influense the mean;

**mean** (7.7 minutes) is higher than **median** (0 minutes) - considering positive delays (50%), the mean shows more realistic picture: 

In [192]:
merged['delay_minutes'].describe()

count    20000.000000
mean         7.703900
std         31.318537
min        -59.000000
25%         -8.000000
50%          0.000000
75%         13.000000
max        522.000000
Name: delay_minutes, dtype: float64